# Standardization

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, year, to_date

import sys, os, time, getpass

sys.path.append("/home/tatiane/lib/")
import pessoal
from pessoal import *

spark = SparkSession.builder \
    .appName("SIM Processing") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## Leitura do arquivo

In [3]:
PATH = '/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data'
ibge_path = f"{PATH}/processed/ibge_codigos/conversion/dtb_brasil_2024_municipios.parquet"
ibge_codigos = spark.read.parquet(ibge_path)

In [4]:
pessoal.completudeSchema(ibge_codigos)

[Stage 1:>                                                          (0 + 1) / 1]

Qtd. registros: 5571 | Quantidade de colunas:  10
root
 |-- UF: long (nullable = true)
 |-- Nome_UF: string (nullable = true)
 |-- Região Geográfica Intermediária: long (nullable = true)
 |-- Nome Região Geográfica Intermediária: string (nullable = true)
 |-- Região Geográfica Imediata: long (nullable = true)
 |-- Nome Região Geográfica Imediata: string (nullable = true)
 |-- Município: long (nullable = true)
 |-- Código Município Completo: long (nullable = true)
 |-- Nome_Município: string (nullable = true)
 |-- Unnamed: 9: string (nullable = true)



### Visualização do dado

In [7]:
ibge_codigos.toPandas().head()

,UF,Nome_UF,Região Geográfica Intermediária,Nome Região Geográfica Intermediária,Região Geográfica Imediata,Nome Região Geográfica Imediata,Município,Código Município Completo,Nome_Município,Unnamed: 9
0,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,15,1100015,Alta Floresta D'Oeste,None
1,11,Rondônia,1102,Ji-Paraná,110005,Cacoal,379,1100379,Alto Alegre dos Parecis,None
2,11,Rondônia,1101,Porto Velho,110002,Ariquemes,403,1100403,Alto Paraíso,None
3,11,Rondônia,1102,Ji-Paraná,110004,Ji-Paraná,346,1100346,Alvorada D'Oeste,None
4,11,Rondônia,1101,Porto Velho,110002,Ariquemes,23,1100023,Ariquemes,None


### Aplicação de mudanças

- Remoção de colunas

In [8]:
ibge_codigos = ibge_codigos.drop('Unnamed: 9')

- Renomeação de colunas

In [10]:
mapping = {
    'UF':'cod_uf',
    'Nome_UF':'nm_uf',
    'Região Geográfica Intermediária':'reg_geo_inter',
    'Nome Região Geográfica Intermediária':'nm_reg_geo_inter',
    'Região Geográfica Imediata':'reg_geo_imed',
    'Nome Região Geográfica Imediata':'nm_reg_geo_imed',
    'Município':'municipio',
    'Código Município Completo':'cod_mun_comp',
    'Nome_Município':'nm_mun'
}

In [11]:
from functools import reduce

def rename_columns(ibge_codigos, mapping):
    return reduce(
        lambda ibge_codigos, col: ibge_codigos.withColumnRenamed(col, mapping[col]),
        mapping,
        ibge_codigos
    )

In [18]:
ibge_codigos = rename_columns(ibge_codigos, mapping)

- Validação

In [19]:
def validate_columns(ibge_codigos, mapping):
    missing = [col for col in mapping.keys() if col not in ibge_codigos.columns]
    if missing:
        print(f"Colunas faltando: {missing}")

In [20]:
pessoal.completudeSchema(ibge_codigos)

Qtd. registros: 5571 | Quantidade de colunas:  9
root
 |-- cod_uf: long (nullable = true)
 |-- nm_uf: string (nullable = true)
 |-- reg_geo_inter: long (nullable = true)
 |-- nm_reg_geo_inter: string (nullable = true)
 |-- reg_geo_imed: long (nullable = true)
 |-- nm_reg_geo_imed: string (nullable = true)
 |-- municipio: long (nullable = true)
 |-- cod_mun_comp: long (nullable = true)
 |-- nm_mun: string (nullable = true)



### Writing the dataset

In [25]:
ibge_codigos.coalesce(1).write.mode("overwrite").parquet("/home/tatiane/repositorios/brazil-health-data-project/data_pipeline/data/processed/ibge_codigos/standardization/")